# NB06 — Edge Sharpness / Gradient Strength

**Physics rationale:** bSSFP banding causes blurring near off-resonance bands.
The optimal offset shows the sharpest myocardium–blood pool boundary.

**Metrics:** `edge_sharpness_global`, `edge_sharpness_myo`, `edge_sharpness_boundary` (myo-pool ring),
`laplacian_variance`, `gradient_energy_myo`, `tenengrad`

**Selector:** `argmax(edge_sharpness_boundary)` vs GT

**Runs on:** Local CPU

## 0. Install Dependencies

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'pydicom', 'numpy', 'pandas', 'matplotlib', 'Pillow', 'openpyxl', 'scipy'])
print('Dependencies OK')

Dependencies OK


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
from scipy import ndimage as ndi
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0f0f1a'
matplotlib.rcParams['axes.facecolor']   = '#1a1a2e'
matplotlib.rcParams['text.color']       = 'white'
%matplotlib inline

DATA_ROOT  = r'E:/CINE-Test1/Dr-Dataset/1st_cohort_SAX'
GT_PATH    = r'E:/CINE-Test1/1stcohortSAX (1).xlsx'
CACHE_PATH = 'outputs/seg_masks_cache.pkl'
OUT_DIR    = 'outputs'
os.makedirs(OUT_DIR, exist_ok=True)
EXCLUDE    = ['Patient-KP']
OFFSETS_13 = [-150 + i*25 for i in range(13)]
print('Config OK')

Config OK


In [3]:
from utils.dicom_loader import load_all_patients
from utils.gt_loader import load_gt
from utils.seg_wrapper import load_seg_cache

all_data  = load_all_patients(DATA_ROOT, verbose=False)
gt_dict   = load_gt(GT_PATH)
seg_cache = load_seg_cache(CACHE_PATH)
print(f'Loaded {len(all_data)} patients')

Loaded 26 patients


## 1. Sharpness Metric Functions

In [4]:
def align_mask(mask, target_shape):
    th, tw = target_shape[:2]
    mh, mw = mask.shape[:2]
    if (mh, mw) == (th, tw): return mask.astype(bool)
    if (mh, mw) == (tw, th): return mask.T.astype(bool)
    from PIL import Image as PILImage
    pil = PILImage.fromarray(mask.astype(np.uint8) * 255)
    pil = pil.resize((tw, th), PILImage.NEAREST)
    return (np.asarray(pil) > 127)

def sobel_magnitude(gray):
    Gx = ndi.sobel(gray, axis=1)
    Gy = ndi.sobel(gray, axis=0)
    return np.sqrt(Gx**2 + Gy**2)

def myo_pool_boundary_mask(myo_mask, pool_mask, dilation_radius=3):
    struct = ndi.generate_binary_structure(2, 2)
    myo_d  = ndi.binary_dilation(myo_mask,  iterations=dilation_radius, structure=struct)
    pool_d = ndi.binary_dilation(pool_mask, iterations=dilation_radius, structure=struct)
    return myo_d & pool_d

def compute_sharpness_metrics(img, myo_mask, pool_mask):
    gray = img.astype(np.float32)
    if gray.max() > 1.0: gray /= gray.max()
    G   = sobel_magnitude(gray)
    lap = ndi.laplace(gray)

    G_myo = G[myo_mask]
    bnd   = myo_pool_boundary_mask(myo_mask, pool_mask, 3)
    G_bnd = G[bnd]
    thresh    = G.mean()

    return {
        'edge_sharpness_global':   round(float(G.mean()), 6),
        'edge_sharpness_myo':      round(float(G_myo.mean()) if G_myo.size > 0 else 0.0, 6),
        'edge_sharpness_boundary': round(float(G_bnd.mean()) if G_bnd.size > 0 else 0.0, 6),
        'laplacian_variance':      round(float(lap.var()), 6),
        'gradient_energy_myo':     round(float((G_myo**2).sum()) if G_myo.size > 0 else 0.0, 4),
        'tenengrad':               round(float((G[G > thresh]**2).sum()) / max(gray.size, 1), 6),
    }

print('Sharpness functions defined.')

Sharpness functions defined.


## 2. Compute for All Patients

In [5]:
if 'all_data' not in globals() or 'seg_cache' not in globals():
    from utils.dicom_loader import load_all_patients
    from utils.gt_loader import load_gt
    from utils.seg_wrapper import load_seg_cache
    all_data  = load_all_patients(DATA_ROOT, verbose=False)
    gt_dict   = load_gt(GT_PATH)
    seg_cache = load_seg_cache(CACHE_PATH)

records = []
for pid in sorted(all_data):
    if pid in EXCLUDE: continue
    data = all_data[pid]
    seg  = seg_cache.get(pid)
    if seg is None: continue

    for i, (img, hz) in enumerate(zip(data['images'], data['offsets_hz'])):
        gray = img.astype(np.float32)
        if gray.max() > 1.0: gray /= gray.max()
        myo_mask  = align_mask(seg.myocardium_mask, gray.shape)
        pool_mask = align_mask(seg.lv_pool_mask,    gray.shape)
        metrics   = compute_sharpness_metrics(gray, myo_mask, pool_mask)
        records.append({'patient_id': pid, 'image_no': i+1, 'offset_hz': hz, **metrics})

df = pd.DataFrame(records)
df.to_csv(f'{OUT_DIR}/edge_metrics.csv', index=False)
print(f'Computed {len(df)} rows. Saved edge_metrics.csv')
df.head(13)

Computed 324 rows. Saved edge_metrics.csv


,patient_id,image_no,offset_hz,edge_sharpness_global,edge_sharpness_myo,edge_sharpness_boundary,laplacian_variance,gradient_energy_myo,tenengrad
0,Patient-BEV,1,-150.0,0.104885,0.135504,0.214429,0.001286,26.7768,0.033904
1,Patient-BEV,2,-125.0,0.106335,0.112044,0.210204,0.001186,17.9375,0.035397
2,Patient-BEV,3,-100.0,0.094347,0.085448,0.144418,0.000912,11.3149,0.028913
3,Patient-BEV,4,-75.0,0.089635,0.068276,0.106979,0.000828,6.8671,0.027112
4,Patient-BEV,5,-50.0,0.086765,0.063906,0.098605,0.000778,6.1328,0.025849
5,Patient-BEV,6,-25.0,0.084473,0.059410,0.092262,0.000727,5.2400,0.024477
6,Patient-BEV,7,0.0,0.085065,0.062950,0.097037,0.000726,5.8523,0.023936
7,Patient-BEV,8,25.0,0.091579,0.072533,0.110197,0.000823,7.3395,0.026316
8,Patient-BEV,9,50.0,0.095732,0.099100,0.134499,0.000898,13.1123,0.027426
9,Patient-BEV,10,75.0,0.098090,0.129464,0.187177,0.001012,21.1808,0.028065


## 3. Sharpness Curves & Gradient Magnitude Visualisation

In [6]:
from utils.vis_utils import plot_metric_curves
import matplotlib.pyplot as plt

for metric_col, title in [
    ('edge_sharpness_boundary', 'Edge Sharpness at Myo-Pool Boundary (most relevant)'),
    ('edge_sharpness_myo',      'Edge Sharpness within Myocardium'),
    ('laplacian_variance',      'Laplacian Variance (Focus Measure)'),
    ('tenengrad',               'Tenengrad Focus Measure'),
]:
    metric_dict = {
        pid: df[df['patient_id']==pid].sort_values('image_no')[metric_col].tolist()
        for pid in sorted(df['patient_id'].unique())
    }
    fig = plot_metric_curves(
        metric_dict=metric_dict, offsets_hz=OFFSETS_13, gt_dict=gt_dict, title=title,
        save_path=f'{OUT_DIR}/edge_{metric_col}_curves.png',
    )
    plt.show()

# Gradient magnitude map demo
DEMO_PID = 'Patient-BGT'
data    = all_data[DEMO_PID]
seg     = seg_cache[DEMO_PID]
gt_no   = gt_dict[DEMO_PID]['image_no']
compare_nos = [max(1, gt_no-2), gt_no, min(len(data['images']), gt_no+2)]

fig, axes = plt.subplots(2, len(compare_nos), figsize=(len(compare_nos)*3.5, 7))
fig.patch.set_facecolor('#0f0f1a')
for col, img_no in enumerate(compare_nos):
    img  = data['images'][img_no-1]
    hz   = data['offsets_hz'][img_no-1]
    gray = img.astype(np.float32)
    if gray.max() > 1.0: gray /= gray.max()
    G = sobel_magnitude(gray)
    color = 'lime' if img_no == gt_no else 'white'
    axes[0, col].imshow(gray, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f'{hz:+.0f} Hz #{img_no}', color=color, fontsize=9)
    axes[0, col].axis('off')
    axes[1, col].imshow(G, cmap='hot', vmin=0, vmax=G.max())
    axes[1, col].set_title('Gradient Magnitude', fontsize=8, color='gray')
    axes[1, col].axis('off')
fig.suptitle(f'{DEMO_PID} — Original (top) + Gradient (bottom), GT = green', color='white', fontsize=10)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/edge_gradient_viz_{DEMO_PID}.png', bbox_inches='tight', dpi=120)
plt.show()

C:\Users\Hp\AppData\Local\Temp\ipykernel_22040\2338730873.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Hp\AppData\Local\Temp\ipykernel_22040\2338730873.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Hp\AppData\Local\Temp\ipykernel_22040\2338730873.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Hp\AppData\Local\Temp\ipykernel_22040\2338730873.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Hp\AppData\Local\Temp\ipykernel_22040\2338730873.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Accuracy Evaluation

In [7]:
from utils.gt_loader import evaluate_predictions
from utils.vis_utils import summary_text

for selector_col, selector_name in [
    ('edge_sharpness_boundary', 'argmax(edge_sharpness_boundary)'),
    ('edge_sharpness_myo',      'argmax(edge_sharpness_myo)'),
    ('laplacian_variance',      'argmax(laplacian_variance)'),
    ('tenengrad',               'argmax(tenengrad)'),
]:
    pred_dict = {}
    for pid in df['patient_id'].unique():
        sub = df[df['patient_id']==pid].sort_values('image_no')
        pred_dict[pid] = int(np.argmax(sub[selector_col].values)) + 1

    res = evaluate_predictions(pred_dict, gt_dict, exclude=EXCLUDE)
    print(summary_text(selector_name, res['top1'], res['pm1'], res['mae_hz'], res['n_valid']))

print('NB06 complete. Proceed to nb07_temporal_stability.ipynb')


  argmax(edge_sharpness_boundary)
  Top-1 accuracy : 0/25 = 0.0%
  ±1 step        : 1/25 = 4.0%
  MAE            : 126.0 Hz


  argmax(edge_sharpness_myo)
  Top-1 accuracy : 0/25 = 0.0%
  ±1 step        : 1/25 = 4.0%
  MAE            : 122.0 Hz


  argmax(laplacian_variance)
  Top-1 accuracy : 1/25 = 4.0%
  ±1 step        : 3/25 = 12.0%
  MAE            : 97.0 Hz


  argmax(tenengrad)
  Top-1 accuracy : 1/25 = 4.0%
  ±1 step        : 4/25 = 16.0%
  MAE            : 84.0 Hz

NB06 complete. Proceed to nb07_temporal_stability.ipynb
